In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [20]:
import requests, zipfile, os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import random

In [21]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"

# Remove any old/corrupted file
if os.path.exists(zip_path):
    os.remove(zip_path)
    print("Removed old zip file.")

# Download with progress
print("Downloading fresh copy...")
response = requests.get(url, stream=True)
response.raise_for_status()  # Will raise error if URL is wrong

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Downloaded: {os.path.getsize(zip_path)/1e6:.2f} MB")

# Verify it's a real ZIP
with zipfile.ZipFile(zip_path) as z:
    print("Valid ZIP! Contains:", z.namelist()[:5])
    print("Extracting...")
    z.extractall(".")

Removed old zip file.
Downloaded: 0.98 MB
Valid ZIP! Contains: ['ml-latest-small/', 'ml-latest-small/links.csv', 'ml-latest-small/tags.csv', 'ml-latest-small/ratings.csv', 'ml-latest-small/README.txt']
Extracting...


In [22]:
# Load ratings file
ratings = pd.read_csv(
    "ml-latest-small/ratings.csv",
    sep=",",
    encoding="latin-1"
)

# Show first few rows
print("\n✅ First five rows:")
print(ratings.head())

# Compute dataset summary
num_ratings = len(ratings)
num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()

print(f"\n📊 Dataset Summary:")
print(f"Row Count for Ratings Dataset: {num_ratings}")
print(f"Number of Users: {num_users}")
print(f"Number of Movies: {num_movies}")


✅ First five rows:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

📊 Dataset Summary:
Row Count for Ratings Dataset: 100836
Number of Users: 610
Number of Movies: 9724


In [23]:
# Preprocess the data
ratings['rating'] = ratings['rating'].astype(float)
high_rated = ratings[ratings['rating'] >= 4]

# Create a user-item matrix
user_item_matrix = high_rated.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# Compute item-item similarity using cosine similarity
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

In [24]:
def greedy_Randomized_Neighborhood_Expansion(group_members, candidate_items, top_n=5, random_factor=0.3):
    selected_items = []
    diversity_scores = []

    # Aggregate group preferences
    group_profile = user_item_matrix.loc[group_members].sum(axis=0)

    for _ in range(top_n):
        # Compute scores for all candidate items
        scores = {}
        for item in candidate_items:
            if item in selected_items:
                continue

            # Marginal gain for the current item
            marginal_gain = group_profile[item]

            # Diversity penalty: similarity with already selected items
            similarity_penalty = sum(item_similarity_df[item][selected_items]) if selected_items else 0

            # Final score = relevance - penalty
            scores[item] = marginal_gain - similarity_penalty

        # Randomized selection: Top candidates based on scores
        sorted_candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_candidates = sorted_candidates[:max(1, int(len(sorted_candidates) * random_factor))]

        # Choose one randomly from the top candidates
        selected_item = random.choice(top_candidates)[0]
        selected_items.append(selected_item)
        diversity_scores.append(scores[selected_item])

    return selected_items

In [25]:
# For an example we have defined user IDs 1, 2, 3 as the group members.
# We can change this at any time to get the output.
group_members = [1, 2, 3]
candidate_items = list(user_item_matrix.columns)
top_recommendations = greedy_Randomized_Neighborhood_Expansion(group_members, candidate_items, top_n=5, random_factor=0.3)

print("Top 5 Diverse Recommendations for the Group:", top_recommendations)


Top 5 Diverse Recommendations for the Group: [1600, 936, 2615, 1390, 3086]
